# 13장. 그레이디언트 부스팅

앞의 트리들이 틀린 부분을 다음 트리가 맞히게 하면 어디까지 좋아지는가.

**준비:** 12장과 같은 CSV 열 개(전력수요·기온 다섯, 서울 아파트 실거래가 넷, 경기도 중국집 하나)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 5분쯤 걸린다.

## 13.1 잔차를 차례로 맞히는 트리

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
# 12.1절에서 만든 함수들이다. 분할을 찾고(best_cut), 트리를 만들고(grow), 트리로 예측한다(ask).
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def best_cut(x, y, min_leaf=1):
    """x를 어느 값에서 둘로 나누면 두 쪽의 제곱 오차 합이 가장 작아지는지 찾는다. (남는 제곱 오차 합, 나누는 값)을 돌려준다."""
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    n = len(ys)
    run, run2 = np.cumsum(ys), np.cumsum(ys ** 2)
    best = (np.inf, None)
    for i in range(min_leaf, n - min_leaf + 1):          # 왼쪽에 i줄, 오른쪽에 n - i줄
        if i == n or xs[i] == xs[i - 1]:
            continue
        left = run2[i - 1] - run[i - 1] ** 2 / i         # 왼쪽 줄들이 왼쪽 평균에서 벗어난 제곱의 합
        right = (run2[-1] - run2[i - 1]) - (run[-1] - run[i - 1]) ** 2 / (n - i)
        if left + right < best[0]:
            best = (left + right, (xs[i - 1] + xs[i]) / 2)
    return best

def grow(X, y, depth, min_leaf=1):
    """특성마다 best_cut을 구해 가장 좋은 특성과 값으로 나눈 뒤, 나뉜 두 쪽에서 같은 일을 반복한다."""
    if depth == 0 or len(y) < 2 * min_leaf:
        return float(y.mean())                           # 잎 노드: 여기에 모인 줄들의 평균이 예측이다
    cuts = [best_cut(X[:, j], y, min_leaf) for j in range(X.shape[1])]
    j = int(np.argmin([c[0] for c in cuts]))
    if cuts[j][1] is None:
        return float(y.mean())
    left = X[:, j] < cuts[j][1]
    return (j, float(cuts[j][1]), grow(X[left], y[left], depth - 1, min_leaf), grow(X[~left], y[~left], depth - 1, min_leaf))

def ask(tree, X):
    """줄마다 뿌리 노드에서 시작해서 조건을 따라 내려가 도착한 잎 노드의 값을 돌려준다."""
    out = np.empty(len(X))
    for i, row in enumerate(X):
        node = tree
        while isinstance(node, tuple):
            node = node[2] if row[node[0]] < node[1] else node[3]
        out[i] = node
    return out

X23, X25 = T23[:, None], T25[:, None]

In [ ]:
said, said25 = np.full(len(Y23), Y23.mean()), np.full(len(Y25), Y23.mean())       # 처음 예측은 평균 하나
print(0, round(rmse(Y23, said), 2), round(rmse(Y25, said25), 2))
stages = []
for step in [1, 2, 3]:
    resid = Y23 - said                               # 지금까지의 예측이 남긴 잔차
    stump = grow(X23, resid, 1)                      # 잔차를 목표로 한, 분할이 하나뿐인 트리
    stages.append((resid, stump))
    said, said25 = said + ask(stump, X23), said25 + ask(stump, X25)
    print(step, (round(stump[1], 2), round(stump[2], 2), round(stump[3], 2)), round(rmse(Y23, said), 2), round(rmse(Y25, said25), 2))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 13-1
xs = np.linspace(-9, 30.5, 800)
fig, axes = plt.subplots(3, 1, figsize=(7, 8.2), sharex=True)
for k, (ax, (resid, stump)) in enumerate(zip(axes, stages), 1):
    ax.scatter(T23, resid, s=9, color=to_rgba(GRAY, 0.7), edgecolor="none")
    ax.plot(xs, ask(stump, xs[:, None]), color=BRICK, lw=1.6)
    ax.axhline(0, color="#44505c", lw=0.7)
    frame(ax, f"{k}번째 트리: {stump[1]:.2f}도에서 나눠 왼쪽 {stump[2]:+.2f}GW, 오른쪽 {stump[3]:+.2f}GW")
    ax.set_ylabel("잔차(GW)", fontsize=8)
    ax.set_ylim(-13, 17)
axes[2].set_xlabel("일평균 기온(°C)", fontsize=8)
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout(h_pad=1.4)
fig.savefig("fig_13_1.png", dpi=300)
plt.show()

In [ ]:
def boost(X, y, rounds, rate, depth, min_leaf=1):
    """평균에서 시작해서, 잔차를 목표로 한 트리를 rounds개 차례로 만든다. 트리의 예측은 rate배만 더한다."""
    said, trees = np.full(len(y), y.mean()), []
    for _ in range(rounds):
        tree = grow(X, y - said, depth, min_leaf)
        said = said + rate * ask(tree, X)
        trees.append(tree)
    return float(y.mean()), trees

def staged(start, trees, rate, X):
    """트리를 하나씩 더할 때마다의 예측. 행이 트리의 수, 열이 예측할 줄이다."""
    return start + rate * np.cumsum([ask(t, X) for t in trees], axis=0)

SHOW = [1, 10, 30, 100, 300, 500]
runs, history = {}, {}
for rate in [1.0, 0.1]:
    start, trees = boost(X23, Y23, 500, rate, 1)
    a, b = staged(start, trees, rate, X23), staged(start, trees, rate, X25)
    runs[rate] = (start, trees)
    history[rate] = ([rmse(Y23, s) for s in a], [rmse(Y25, s) for s in b])
    print(rate, [(round(history[rate][0][k - 1], 2), round(history[rate][1][k - 1], 2)) for k in SHOW])

In [ ]:
# 그림 13-3
curve = staged(*runs[0.1], 0.1, xs[:, None])
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.scatter(T23, Y23, s=9, color=to_rgba(GRAY, 0.6), edgecolor="none")
for k, col, style in [(10, TEAL, "-"), (50, NAVY, "-"), (300, BRICK, "-")]:
    ax.plot(xs, curve[k - 1], color=col, ls=style, lw=1.7, label=f"트리 {k}개")
ax.set_xlabel("일평균 기온(°C)", fontsize=8)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=8)
ax.set_ylim(54, 90)
ax.legend(frameon=False, fontsize=8, loc="upper center", ncol=3)
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 13-4
fig, ax = plt.subplots(figsize=(7, 4.2))
ks = np.arange(1, 501)
ax.plot(ks, history[1.0][1], color=BRICK, lw=1.8, label="학습률 1.0, 2025년(시험 자료)")
ax.plot(ks, history[1.0][0], color=BRICK, lw=1.3, ls=(0, (4, 2)), label="학습률 1.0, 2023년(훈련 자료)")
ax.plot(ks, history[0.1][1], color=NAVY, lw=1.8, label="학습률 0.1, 2025년(시험 자료)")
ax.plot(ks, history[0.1][0], color=NAVY, lw=1.3, ls=(0, (4, 2)), label="학습률 0.1, 2023년(훈련 자료)")
ax.set_xscale("log"); ax.set_xlim(0.9, 550); ax.set_ylim(1.5, 6.9)
plain_ticks(ax.xaxis)
ax.set_xlabel("트리의 수(로그 축)", fontsize=8)
ax.set_ylabel("RMSE(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_4.png", dpi=300)
plt.show()

## 13.2 부스팅과 경사하강법: 학습률과 손실 함수

In [ ]:
rows, tests = [], {}
for rate, depth, rounds in [(1.0, 1, 500), (0.3, 1, 500), (0.1, 1, 500), (0.03, 1, 1500), (1.0, 2, 500), (0.1, 2, 500)]:
    start, trees = boost(X23, Y23, rounds, rate, depth)
    a, b = staged(start, trees, rate, X23), staged(start, trees, rate, X25)
    test = np.array([rmse(Y25, s) for s in b])
    tests[(rate, depth)] = test
    rows.append((rate, depth, int(test.argmin()) + 1, round(float(test.min()), 2), round(rmse(Y23, a[-1]), 2), round(float(test[-1]), 2)))
print(pd.DataFrame(rows, columns=["학습률", "깊이", "최적 트리 수", "최소 시험 RMSE", "끝 훈련 RMSE", "끝 시험 RMSE"]).to_string(index=False))

In [ ]:
# 그림 13-5
fig, ax = plt.subplots(figsize=(7, 4.2))
for (rate, col) in [(1.0, BRICK), (0.3, TEAL), (0.1, NAVY), (0.03, GRAY)]:
    t = tests[(rate, 1)]
    ax.plot(np.arange(1, len(t) + 1), t, color=col, lw=1.7, label=f"학습률 {rate:g}")
    ax.scatter([t.argmin() + 1], [t.min()], s=34, color=col, zorder=5)
ax.set_xscale("log"); ax.set_xlim(0.9, 1700); ax.set_ylim(3.05, 6.8)
plain_ticks(ax.xaxis)
ax.set_xlabel("트리의 수(로그 축)", fontsize=8)
ax.set_ylabel("2025년 RMSE(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.text(1.05, 3.14, "점: 각 학습률에서 시험 RMSE가 가장 작은 곳", fontsize=8, color="#5b6570")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_5.png", dpi=300)
plt.show()

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(actual, prob):
    prob = np.clip(prob, 1e-12, 1 - 1e-12)
    return float(-(actual * np.log(prob) + (1 - actual) * np.log(1 - prob)).mean())

def brier(actual, prob):
    return float(((prob - actual) ** 2).mean())

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

y23, y25 = (Y23 >= 74).astype(float), (Y25 >= 74).astype(float)        # 11장의 문제: 일평균 수요가 74GW 이상인 날인가
start = float(np.log(y23.mean() / (1 - y23.mean())))                   # 처음 예측은 기준율의 로그 승산
score, score25, rows, stumps = np.full(len(y23), start), np.full(len(y25), start), [], []
for k in range(1, 301):
    stump = grow(X23, y23 - sigmoid(score), 1)       # 목표가 '레이블 - 예측 확률'로 바뀌었을 뿐이다
    stumps.append(stump)
    score, score25 = score + ask(stump, X23), score25 + ask(stump, X25)
    if k in (1, 10, 30, 100, 300):
        rows.append((k, round(log_loss(y23, sigmoid(score)), 4), round(log_loss(y25, sigmoid(score25)), 4),
                     round(brier(y25, sigmoid(score25)), 4), round(auc(y25, score25), 3)))
print(pd.DataFrame(rows, columns=["트리 수", "2023년 로그 손실", "2025년 로그 손실", "2025년 브라이어", "2025년 AUC"]).to_string(index=False))

In [ ]:
# 그림 13-6
steps = start + np.cumsum([ask(s, xs[:, None]) for s in stumps], axis=0)
jitter = np.random.default_rng(0).uniform(-0.03, 0.03, len(y23))
fig, ax = plt.subplots(figsize=(7, 4.1))
ax.scatter(T23, y23 + jitter, s=11, color=to_rgba(GRAY, 0.6), edgecolor="none")
for k, col in [(10, TEAL), (100, NAVY), (300, BRICK)]:
    ax.plot(xs, sigmoid(steps[k - 1]), color=col, lw=1.6, label=f"트리 {k}개")
ax.axhline(0, color=GRAY, lw=0.8); ax.axhline(1, color=GRAY, lw=0.8)
ax.set_xlim(-9.5, 31.5); ax.set_ylim(-0.1, 1.12)
ax.set_xlabel("일평균 기온(°C)", fontsize=8)
ax.set_ylabel("수요가 많은 날일 확률", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_6.png", dpi=300)
plt.show()

## 13.3 트리의 수 고르기: 조기 종료

In [ ]:
def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
def seven(df):
    """6장의 특성을 가변수로 바꾸지 않고 숫자 그대로 준다. 업력, 위도, 경도, 지번폐업, 주변, 주변폐업, 시군 번호."""
    return np.column_stack([df["업력"], df["위도"].fillna(0), df["경도"].fillna(0), df["지번폐업"],
                            df["주변"].fillna(-1), df["주변폐업"].fillna(-1), pd.Index(SIGUN).get_indexer(df["시군"])])

year = train["기준일"].dt.year.to_numpy()
fit_on, tune_on = year < 2020, year == 2020
R, Rt = seven(train), seven(test)

from sklearn.ensemble import HistGradientBoostingClassifier

base = float(ytr[fit_on].mean())
flat = log_loss(ytr[tune_on], np.full(tune_on.sum(), base))
print(round(flat, 4))
curves, made_curves, rows = {}, {}, []
for name, depth in [("기본 크기(잎 31개까지)", None), ("깊이 3", 3), ("깊이 1", 1)]:
    m = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=500, max_depth=depth, early_stopping=False, random_state=0).fit(R[fit_on], ytr[fit_on])
    made = [log_loss(ytr[fit_on], s[:, 1]) for s in m.staged_predict_proba(R[fit_on])]
    seen = [log_loss(ytr[tune_on], s[:, 1]) for s in m.staged_predict_proba(R[tune_on])]
    curves[name], made_curves[name] = seen, made
    best = int(np.argmin(seen))
    rows.append((name, *[f"{made[k - 1]:.3f}/{seen[k - 1]:.3f}" for k in (10, 30, 100, 500)]))
    print(name, best + 1, round(seen[best], 4))
print(pd.DataFrame(rows, columns=["트리의 크기", "10개", "30개", "100개", "500개"]).to_string(index=False))
print(next(k + 1 for k, v in enumerate(curves["기본 크기(잎 31개까지)"]) if k > 8 and v > flat))

In [ ]:
# 그림 13-7
fig, ax = plt.subplots(figsize=(7, 4.3))
looks = [("기본 크기(잎 31개까지)", BRICK), ("깊이 3", NAVY), ("깊이 1", TEAL)]
for name, col in looks:
    seen = np.array(curves[name])
    ax.plot(range(1, 501), seen, color=col, lw=1.7, label=f"{name}: {seen.argmin() + 1}개에서 {seen.min():.4f}")
    ax.plot(range(1, 501), made_curves[name], color=col, lw=1.1, ls=(0, (4, 2)))
    ax.scatter([seen.argmin() + 1], [seen.min()], s=34, color=col, zorder=5)
ax.axhline(flat, color=GRAY, lw=1.1, ls=(0, (1, 2)), label=f"기준율로만 예측 {flat:.4f}")
ax.text(1.05, 0.4335, "점선(색): 같은 모델의 훈련 자료 로그 손실", fontsize=8, color="#5b6570", ha="left")
ax.set_xscale("log"); ax.set_xlim(0.9, 550); ax.set_ylim(0.43, 0.505)
plain_ticks(ax.xaxis)
ax.set_xlabel("트리의 수(로그 축)", fontsize=8)
ax.set_ylabel("로그 손실(실선: 2020년 검증 자료)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_7.png", dpi=300)
plt.show()

In [ ]:
plain = HistGradientBoostingClassifier(random_state=0).fit(R[fit_on], ytr[fit_on])          # 설정을 하나도 주지 않았다
said = plain.predict_proba(R[tune_on])[:, 1]
print(plain.get_params()["early_stopping"], plain.get_params()["validation_fraction"], plain.get_params()["n_iter_no_change"],
      plain.n_iter_, round(log_loss(ytr[tune_on], said), 4), round(auc(ytr[tune_on], said), 3))

picked = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=33, max_depth=3, early_stopping=False, random_state=0).fit(R[fit_on], ytr[fit_on])
said = picked.predict_proba(R[tune_on])[:, 1]
print(round(log_loss(ytr[tune_on], said), 4), round(brier(ytr[tune_on], said), 4), round(auc(ytr[tune_on], said), 3), round(float(said.min()), 2), round(float(said.max()), 2))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

def exam(name, said):
    print(f"{name}: AUC {auc(y, said):.3f} 브라이어 {brier(y, said):.4f} 로그 손실 {log_loss(y, said):.4f}")

print(f"기준율로만 예측: 브라이어 {brier(y, base):.4f} 로그 손실 {log_loss(y, np.full(len(y), base)):.4f}")
t_logit = LogisticRegression(C=1e6, max_iter=20000).fit(X[fit_on], ytr[fit_on]).predict_proba(Xt)[:, 1]
t_boost = picked.predict_proba(Rt)[:, 1]
exam("로지스틱, 특성 41개", t_logit)
exam("랜덤 포레스트, 잎에 200줄 이상", RandomForestClassifier(300, min_samples_leaf=200, n_jobs=-1, random_state=0).fit(R[fit_on], ytr[fit_on]).predict_proba(Rt)[:, 1])
exam("부스팅, 깊이 3인 트리 33개", t_boost)
long = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=500, early_stopping=False, random_state=0).fit(R[fit_on], ytr[fit_on])
exam("부스팅, 기본 크기의 트리 500개", long.predict_proba(Rt)[:, 1])

rng = np.random.default_rng(0)
gaps = []
for _ in range(2000):
    pick = rng.integers(0, len(y), len(y))               # 8.2절의 짝지은 부트스트랩
    gaps.append(auc(y[pick], t_logit[pick]) - auc(y[pick], t_boost[pick]))
print(f"로지스틱 - 부스팅(깊이 3, 33개): AUC 차이 {np.mean(gaps):+.3f} 95% 구간 [{np.quantile(gaps, 0.025) + 0:+.3f}, {np.quantile(gaps, 0.975) + 0:+.3f}]")

## 13.4 선형 모델의 잔차에 부스팅하기

In [ ]:
import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
a23, a24, a25 = v[v["연"] == 2023], v[v["연"] == 2024], v[v["연"] == 2025]
both = v[v["연"] <= 2024]

def off(actual_log, predicted_log):
    """오차율의 중앙값(%). 2장의 off와 같은 지표를 로그 가격에서 계산한다."""
    return round(float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100), 1)

def log_rmse(actual_log, predicted_log):
    return round(float(np.sqrt(((predicted_log - actual_log) ** 2).mean())), 4)

def table_of(df, ref, month=False):
    """면적, 층, 건축년도는 숫자 그대로 쓴다. 동과 단지는 ref에서 가나다순으로 매긴 번호로 바꾼다. 목록에 없는 이름은 -1이 된다."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    cols = [df["면적"], df["층"], df["건축년도"], dongs.get_indexer(df["동이름"]), complexes.get_indexer(df["단지"])]
    return np.column_stack(cols + ([df["달"]] if month else []))

from scipy import sparse
from sklearn.linear_model import Ridge

def design(df, ref):
    """10장의 특성 행렬에 계약한 달을 수치 특성 하나로 추가했다. 수치 특성 8개에 동과 단지의 가변수."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    num = np.column_stack([np.log(df["면적"]), df["층"], *((np.clip(df["연식"], 0, 59) // 10 == k).astype(float) for k in range(1, 6)), df["달"]])
    blocks = [sparse.csr_matrix(num)]
    for col, cats in [("동이름", dongs), ("단지", complexes)]:
        where_ = cats.get_indexer(df[col])
        known = where_ >= 0
        blocks.append(sparse.csr_matrix((np.ones(known.sum()), (np.where(known)[0], where_[known])), shape=(len(df), len(cats))))
    return sparse.hstack(blocks).tocsr()

from sklearn.ensemble import HistGradientBoostingRegressor

p23, p24, pb, p25 = (np.log(d_["억"].to_numpy()) for d_ in (a23, a24, both, a25))
A23, A24 = table_of(a23, a23, month=True), table_of(a24, a23, month=True)          # 설정을 고를 때: 2023년으로 추정해서 2024년으로 평가
AB, A25 = table_of(both, both, month=True), table_of(a25, both, month=True)        # 마지막에: 2024년까지로 추정해서 2025년으로 평가
quarter = (a25["날짜"].dt.month.to_numpy() - 1) // 3
results = {}

def report(name, said):
    results[name] = (off(p25, said), log_rmse(p25, said), [round(float((said - p25)[quarter == q].mean()) * 100, 1) for q in range(4)])
    print(name, *results[name])

In [ ]:
g = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=2000, early_stopping=False, random_state=0).fit(A23, p23)
steps = list(g.staged_predict(A24))
print([(k, off(p24, steps[k - 1]), log_rmse(p24, steps[k - 1])) for k in (100, 300, 1000, 2000)])
alone = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=2000, early_stopping=False, random_state=0).fit(AB, pb)
report("부스팅", alone.predict(A25))
print(both["단지"].nunique(), both["동"].nunique())

In [ ]:
import lightgbm as lgb

SET = dict(objective="regression", learning_rate=0.1, num_leaves=31, min_data_in_leaf=20, cat_smooth=10, verbose=-1, seed=0)
seen = {}
made = lgb.Dataset(A23, p23, categorical_feature=[3, 4], free_raw_data=False)       # 동과 단지가 범주형 특성이라고 알려 준다
lgb.train(SET, made, 2500, valid_sets=[lgb.Dataset(A24, p24, reference=made)], callbacks=[lgb.record_evaluation(seen)])
track = np.sqrt(seen["valid_0"]["l2"])
print([(k, round(float(track[k - 1]), 4)) for k in (100, 300, 1000, 2500)])
named = lgb.train(SET, lgb.Dataset(AB, pb, categorical_feature=[3, 4]), 2500)
report("부스팅(범주형 처리)", named.predict(A25))

In [ ]:
line23 = Ridge(alpha=0.01, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(design(a23, a23), p23)
r24 = line23.predict(design(a24, a23))
print(off(p24, r24), log_rmse(p24, r24))
line = Ridge(alpha=0.01, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(design(both, both), pb)
r25 = line.predict(design(a25, both))
report("릿지", r25)

In [ ]:
left23 = p23 - line23.predict(design(a23, a23))                                     # 릿지가 2023년에 남긴 잔차
fix = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=1000, early_stopping=False, random_state=0).fit(A23, left23)
print(round(float(left23.std()), 4), [(k, off(p24, r24 + s), log_rmse(p24, r24 + s)) for k, s in enumerate(fix.staged_predict(A24), 1) if k in (10, 100, 300, 1000)])

leftb = pb - line.predict(design(both, both))
mend = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=1000, early_stopping=False, random_state=0).fit(AB, leftb)
both25 = r25 + mend.predict(A25)
report("릿지 + 부스팅", both25)
new = ~a25["단지"].isin(both["단지"]).to_numpy()
print(off(p25[new], both25[new]), off(p25[~new], both25[~new]), off(p25[new], r25[new]), off(p25[~new], r25[~new]))

In [ ]:
# 그림 13-8
fig, ax = plt.subplots(figsize=(7, 4.1))
pos = np.arange(4)
looks = [("부스팅", to_rgba(BRICK, 0.55), BRICK), ("부스팅(범주형 처리)", to_rgba(GRAY, 0.5), "#5b6570"), ("릿지", to_rgba(NAVY, 0.22), NAVY), ("릿지 + 부스팅", to_rgba(TEAL, 0.45), TEAL)]
for k, (name, fc, ec) in enumerate(looks):
    vals = np.array(results[name][2])
    ax.bar(pos + (k - 1.5) * 0.2, vals, width=0.185, color=fc, edgecolor=ec, lw=0.9, label=name)
    for x_, val in zip(pos + (k - 1.5) * 0.2, vals):
        ax.text(x_, val - 0.3, f"{val:.1f}", fontsize=7.2, color=ec, ha="center", va="top")
ax.axhline(0, color="#44505c", lw=0.8)
ax.set_xticks(pos, ["2025년 1분기", "2분기", "3분기", "4분기"])
ax.set_ylim(-14, 1.2)
ax.set_ylabel("예측 - 실제의 평균(로그 가격 × 100, 약 %)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_8.png", dpi=300)
plt.show()

In [ ]:
one = a25.iloc[[0]]                                  # 12.4절에서 쓴 것과 같은 거래다
months = np.arange(1, 37)
moved = pd.concat([one] * len(months)).assign(달=months)
M = table_of(moved, both, month=True)
path = {"부스팅": np.exp(alone.predict(M)), "릿지": np.exp(line.predict(design(moved, both))),
        "릿지 + 부스팅": np.exp(line.predict(design(moved, both)) + mend.predict(M))}
print({k: v_[[23, 24, 29, 35]].round(2).tolist() for k, v_ in path.items()})

In [ ]:
# 그림 13-9
fig, ax = plt.subplots(figsize=(7, 4.1))
ax.axvspan(24.5, 36.6, color=to_rgba(BRICK, 0.08), lw=0)
for (name, values), col, style in zip(path.items(), [BRICK, NAVY, TEAL], ["-", (0, (4, 2)), "-"]):
    ax.plot(months, values, color=col, ls=style, lw=1.8, label=name)
ax.axvline(24.5, color=GRAY, lw=1, ls=(0, (4, 2)))
ax.text(30.5, 12.5, "훈련 자료에 없는 기간\n(2025년)", fontsize=8, color=BRICK, ha="center", linespacing=1.5)
ax.set_xticks([1, 7, 13, 19, 25, 31, 36], ["2023.1", "2023.7", "2024.1", "2024.7", "2025.1", "2025.7", "2025.12"])
ax.set_xlim(0.5, 36.6)
ax.set_xlabel("계약한 달(다른 특성은 그대로 두고 이 값만 바꿨다)", fontsize=8)
ax.set_ylabel("같은 집에 대한 예측 가격(억 원)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_13_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 13-10
past = [("2장: 선형 회귀 + 목표 인코딩", 10.7), ("10장: 릿지(단지 가변수)", 9.9), ("12장: 랜덤 포레스트(달 추가)", 7.5)]
now = [("부스팅(달 추가)", results["부스팅"][0]), ("릿지(달 추가)", results["릿지"][0]), ("부스팅(범주형 처리)", results["부스팅(범주형 처리)"][0]), ("릿지 + 부스팅", results["릿지 + 부스팅"][0])]
rows_ = past + now
fig, ax = plt.subplots(figsize=(7, 4.1))
ypos = np.arange(len(rows_))[::-1]
cols = [to_rgba(GRAY, 0.5)] * len(past) + [to_rgba(NAVY, 0.22)] * (len(now) - 1) + [to_rgba(BRICK, 0.55)]
edges = ["#5b6570"] * len(past) + [NAVY] * (len(now) - 1) + [BRICK]
ax.barh(ypos, [val for _, val in rows_], color=cols, edgecolor=edges, lw=0.9, height=0.62)
for yy, (_, val), ec in zip(ypos, rows_, edges):
    ax.text(val + 0.15, yy, f"{val:.1f}%", fontsize=8, color=ec, va="center")
ax.set_yticks(ypos, [name for name, _ in rows_], fontsize=8)
ax.set_xlim(0, 12.5)
ax.set_xlabel("2025년 오차율의 중앙값(%)", fontsize=8)
frame(ax, grid="x")
ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig("fig_13_10.png", dpi=300)
plt.show()